# importlib_resources Package Testing

This notebook tests the importlib_resources package, which reads data files that ship inside Python packages. Every cell calls the `importlib_resources` package itself, not the standard-library `importlib.resources`.

## Example 1: Basic Resource Access

In [ ]:
import email
import importlib_resources

# files() returns a Traversable for a package, given its name or the module itself.
package = importlib_resources.files("importlib_resources")
print(f"Package files: {package}")
assert package.is_dir()
names = sorted(item.name for item in package.iterdir())
print(f"Contents: {names[:8]}")
assert "__init__.py" in names

init = package / "__init__.py"
assert init.is_file() and not init.is_dir()
assert "files" in init.read_text(encoding="utf-8")

email_resources = importlib_resources.files(email)
print(f"\nEmail package resources: {email_resources}")
assert (email_resources / "message.py").is_file()

try:
    importlib_resources.files("no_such_package_for_this_notebook")
except ModuleNotFoundError as error:
    print(f"Missing package raises ModuleNotFoundError: {error}")
else:
    raise AssertionError("files() accepted a package that does not exist")

## Example 2: Package Resource Discovery

In [ ]:
import importlib_resources

# Discover resources in standard library packages
packages_to_test = ["json", "urllib", "http", "xml"]

for package_name in packages_to_test:
    package_files = importlib_resources.files(package_name)
    print(f"\n{package_name} package:")
    print(f"  Location: {package_files}")

    contents = list(package_files.iterdir())
    file_names = sorted(f.name for f in contents if f.is_file())
    dir_names = sorted(f.name for f in contents if f.is_dir() and f.name != "__pycache__")
    print(f"  Files: {file_names[:5]}")
    if dir_names:
        print(f"  Subdirectories: {dir_names[:3]}")
    assert "__init__.py" in file_names, (package_name, file_names)

# Subpackages are directories inside the parent package.
assert (importlib_resources.files("xml") / "etree").is_dir()
assert (importlib_resources.files("xml") / "etree" / "ElementTree.py").is_file()

## Example 3: Reading Resource Contents

In [ ]:
import importlib
import importlib_resources
import pathlib
import sys
import tempfile

# A throwaway package with data files, put on sys.path so it can be imported.
site = tempfile.mkdtemp()
package_dir = pathlib.Path(site, "demo_resources_pkg")
(package_dir / "data").mkdir(parents=True)
(package_dir / "__init__.py").write_text("")
(package_dir / "data" / "greeting.txt").write_text("Hello from importlib_resources!\n", encoding="utf-8")
(package_dir / "data" / "blob.bin").write_bytes(bytes(range(8)))
sys.path.insert(0, site)
importlib.invalidate_caches()

try:
    # Traversable API
    greeting = importlib_resources.files("demo_resources_pkg") / "data" / "greeting.txt"
    content = greeting.read_text(encoding="utf-8")
    print(f"File content: {content!r}")
    assert content == "Hello from importlib_resources!\n"
    blob = importlib_resources.files("demo_resources_pkg").joinpath("data", "blob.bin").read_bytes()
    print(f"Binary content length: {len(blob)} bytes")
    assert blob == bytes(range(8))

    # Functional API: package name followed by the path inside it
    assert importlib_resources.read_text("demo_resources_pkg", "data", "greeting.txt", encoding="utf-8") == content
    assert importlib_resources.read_binary("demo_resources_pkg", "data", "blob.bin") == blob
    with importlib_resources.open_text("demo_resources_pkg", "data", "greeting.txt", encoding="utf-8") as handle:
        assert handle.readline() == content
    assert importlib_resources.is_resource("demo_resources_pkg", "data", "greeting.txt")
    assert not importlib_resources.is_resource("demo_resources_pkg", "data")
    listed = sorted(item.name for item in importlib_resources.files("demo_resources_pkg").joinpath("data").iterdir())
    print(f"Resources in data/: {listed}")
    assert listed == ["blob.bin", "greeting.txt"]
finally:
    sys.path.remove(site)
    sys.modules.pop("demo_resources_pkg", None)

## Example 4: Resource Path Operations

In [ ]:
import importlib
import importlib_resources
import pathlib
import sys
import tempfile
import zipfile

# as_file() gives a real file system path for a resource.
init = importlib_resources.files("importlib_resources") / "__init__.py"
with importlib_resources.as_file(init) as path:
    print(f"On-disk resource: {path}")
    assert isinstance(path, pathlib.Path) and path.is_file()

# For a package imported from a zip file, as_file() extracts a temporary copy
# and removes it again when the block ends.
site = tempfile.mkdtemp()
archive = pathlib.Path(site, "zipped_resources.zip")
with zipfile.ZipFile(archive, "w") as zf:
    zf.writestr("zipped_resources_pkg/__init__.py", "")
    zf.writestr("zipped_resources_pkg/config.json", '{"answer": 42}')
sys.path.insert(0, str(archive))
importlib.invalidate_caches()

try:
    config = importlib_resources.files("zipped_resources_pkg") / "config.json"
    assert config.is_file()
    assert config.read_text(encoding="utf-8") == '{"answer": 42}'
    with importlib_resources.as_file(config) as path:
        print(f"Temporary copy of a zipped resource: {path}")
        assert path.read_text(encoding="utf-8") == '{"answer": 42}'
    assert not path.exists(), "the temporary copy was not removed"

    # path() is the functional form of as_file()
    with importlib_resources.path("zipped_resources_pkg", "config.json") as path:
        assert path.read_text(encoding="utf-8") == '{"answer": 42}'
finally:
    sys.path.remove(str(archive))
    sys.modules.pop("zipped_resources_pkg", None)

## Example 5: Advanced Resource Operations

In [ ]:
import importlib_resources

# Walk a package tree with the Traversable API alone.
def explore_package_structure(package_name):
    files = importlib_resources.files(package_name)
    items = [item for item in files.iterdir() if item.name != "__pycache__"]
    py_files = sorted(f.name for f in items if f.is_file() and f.name.endswith(".py"))
    other_files = sorted(f.name for f in items if f.is_file() and not f.name.endswith(".py"))
    subdirs = sorted(f.name for f in items if f.is_dir())

    print(f"\nExploring {package_name}:")
    print(f"  Python files: {len(py_files)}")
    print(f"  Other files: {len(other_files)}")
    print(f"  Subdirectories: {len(subdirs)}")
    if py_files:
        print(f"  Example .py files: {py_files[:3]}")
    return py_files, other_files, subdirs

# Test with various packages
for pkg in ["collections", "logging", "urllib"]:
    py_files, _, _ = explore_package_structure(pkg)
    assert "__init__.py" in py_files, (pkg, py_files)

py_files, _, subdirs = explore_package_structure("importlib_resources")
assert "__init__.py" in py_files and "abc.py" in py_files, py_files